# Statistics for Data Analysts/Engineers — Module 10: ANOVA

In Module 8, the t-test compared the means of TWO groups. But what about **more than
two** groups -- like our four regions? In this module we'll formalize the satisfaction
difference between regions that we saw visually back in Module 2 (the box plot) -- time
for a formal test.

## Table of contents
1. [Why not run many t-tests at once?](#sec1)
2. [The logic of ANOVA: between-group vs. within-group variance](#sec2)
3. [One-way ANOVA in practice](#sec3)
4. [ANOVA assumptions](#sec4)
5. [Post-hoc test: Tukey HSD](#sec5)
6. [Effect size: eta squared](#sec6)
7. [Summary and exercises](#sec7)


Every module in this course starts by generating the same synthetic dataset -- thanks to a fixed random seed (`np.random.default_rng(42)`), the numbers always come out identical, so you can open any module independently of the others, even after restarting the kernel.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 400

city = rng.choice(
    ["New York", "Chicago", "Austin", "Seattle", "Denver"],
    size=n, p=[0.30, 0.25, 0.15, 0.15, 0.15],
)
region = rng.choice(["North", "South", "East", "West"], size=n)
age = rng.integers(18, 70, size=n)

# monthly spend: right-skewed distribution (typical for spending data)
monthly_spend = rng.lognormal(mean=5.2, sigma=0.5, size=n).round(2)

# satisfaction correlated with spend + a region effect (deliberately different) + random
# noise, clipped to a 1-10 scale -- this gives Module 10 (ANOVA) a real between-group gap
region_effect = {"North": 0.0, "South": -0.5, "East": 0.3, "West": 0.8}
satisfaction = np.clip(
    monthly_spend / 45
    + np.array([region_effect[r] for r in region])
    + rng.normal(0, 1.5, size=n),
    1, 10,
).round(1)

# A/B campaign group -- with a REAL injected effect on conversion (group B better)
group = rng.choice(["A", "B"], size=n)
conversion_prob = np.where(group == "A", 0.10, 0.20)
converted = rng.binomial(1, conversion_prob)

# spend before/after a loyalty-program rollout (paired data)
spend_before = rng.normal(220, 40, size=n).round(2)
program_effect = rng.normal(15, 10, size=n)
spend_after = (spend_before + program_effect).round(2)

customers = pd.DataFrame({
    "customer_id": np.arange(1, n + 1),
    "city": city,
    "region": region,
    "age": age,
    "monthly_spend": monthly_spend,
    "satisfaction": satisfaction,
    "campaign_group": group,
    "converted": converted,
    "spend_before": spend_before,
    "spend_after": spend_after,
})
customers.head()


<a id="sec1"></a>
## 1. Why not run many t-tests at once?

With 4 regions, we might think: just run a t-test for every pair of regions separately
(North-South, North-East, North-West, South-East, ...) -- for 4 groups that's `6` pairs.
The problem: each individual t-test carries `alpha=0.05` risk of a Type I error
(Module 7), but with 6 INDEPENDENT tests at once, the probability that AT LEAST ONE of
them turns up falsely significant grows well above 5%. This is the so-called **multiple
comparisons problem**.

In [ ]:
alpha = 0.05
num_tests = 6
# probability of at least one false alarm across 6 independent tests
p_at_least_one_false_positive = 1 - (1 - alpha) ** num_tests
print(f"Risk of at least one false alarm: {p_at_least_one_false_positive:.3f}")


Nearly 27% instead of the assumed 5%! **ANOVA** (*Analysis of Variance*) solves this by
testing ALL groups with a SINGLE test, at one, controlled `alpha` level.

<a id="sec2"></a>
## 2. The logic of ANOVA: between-group vs. within-group variance

The name "analysis of variance" sounds misleading -- ANOVA tests differences in MEANS, but
it does so by comparing TWO sources of variance:

- **Between-group variance** — how much the means of the individual groups differ from
  each other.
- **Within-group variance** — how spread out the observations are inside each single
  group.

The intuition: if the groups really differ, the variance BETWEEN groups should be large
compared to the "natural noise" WITHIN groups. The `F` test statistic is exactly this
ratio:

```
F = between-group variance / within-group variance
```

`F` close to `1` suggests no difference (both variances similar in size), `F` much
larger than `1` suggests a real difference between group means.

- H0: all group means are equal (`mu_1 = mu_2 = mu_3 = mu_4`)
- H1: at least one group mean differs from the others

<a id="sec3"></a>
## 3. One-way ANOVA in practice

Let's formally test the satisfaction difference between regions that we saw in the box
plot in Module 2:

In [ ]:
from scipy import stats

region_groups = [data["satisfaction"].values for _, data in customers.groupby("region")]

f_statistic, p_value = stats.f_oneway(*region_groups)
print(f"F = {f_statistic:.3f}")
print(f"p-value = {p_value:.6f}")

alpha = 0.05
print("Reject H0" if p_value <= alpha else "Fail to reject H0")


The same result, with a fuller report (including an ANOVA table), can be obtained
through `statsmodels` using formulas (similar to the regression in Module 12):

In [ ]:
import statsmodels.api as sm
from statsmodels.formula.api import ols

model = ols("satisfaction ~ C(region)", data=customers).fit()
anova_table = sm.stats.anova_lm(model, typ=2)
anova_table


> 📊 **C(region) means 'treat as categorical'**
>
> `C(...)` in a `statsmodels` formula (*categorical*) tells the model to treat `region` as a categorical variable (a separate effect for each value), not a number. For text columns like `region`, `statsmodels` usually figures this out on its own, but explicit `C()` is a good habit, especially for columns that LOOK like numbers (e.g. if we'd coded regions as 1/2/3/4).

<a id="sec4"></a>
## 4. ANOVA assumptions

Like the t-test, ANOVA assumes an approximately normal distribution within each group
and **homogeneity of variance** -- that the variance is similar across all groups. Let's
check with the same tool from Module 8 (Levene), this time for more than two groups at
once:

In [ ]:
print("Levene (all 4 regions):", stats.levene(*region_groups))


> ⚠️ **When variances aren't homogeneous: Welch ANOVA**
>
> Classic ANOVA (like `f_oneway`) assumes equal variances, just like the classic t-test. When Levene indicates clearly unequal variances, the counterpart to Module 8's Welch's test is **Welch ANOVA** (`pingouin.welch_anova` in the `pingouin` library, outside standard `scipy`/`statsmodels`) -- a safer choice when variance clearly differs between groups.

<a id="sec5"></a>
## 5. Post-hoc test: Tukey HSD

ANOVA only answers "does AT LEAST ONE group differ?" -- it doesn't say WHICH specific
pairs of groups differ from each other. That's what **post-hoc tests** are for ("after
the fact"), run ONLY once ANOVA has come out significant. The most popular is **Tukey
HSD** (*Honestly Significant Difference*) -- it compares every pair of groups while
controlling the overall Type I error risk (exactly the problem from section 1).

In [ ]:
from statsmodels.stats.multicomp import pairwise_tukeyhsd

tukey_result = pairwise_tukeyhsd(
    endog=customers["satisfaction"], groups=customers["region"], alpha=0.05
)
print(tukey_result)


The `reject` column tells you directly which pair of regions has a statistically
significant difference (`True`) -- exactly the information plain ANOVA didn't give us,
now obtained in a safe, corrected way.

In [ ]:
fig = tukey_result.plot_simultaneous()


<a id="sec6"></a>
## 6. Effect size: eta squared

**Eta squared** (`eta^2`) is ANOVA's counterpart to Cohen's d (Module 8) and Cramér's V
(Module 9) -- it measures what PERCENTAGE of the total variance in satisfaction can be
explained by region membership:

```
eta^2 = between-group sum of squares / total sum of squares
```

Conventional thresholds (similar to Cohen's d): `0.01` small effect, `0.06` medium,
`0.14`+ large.

In [ ]:
between_group_ss = anova_table["sum_sq"]["C(region)"]
total_ss = anova_table["sum_sq"].sum()
eta_squared = between_group_ss / total_ss

print(f"eta^2 = {eta_squared:.3f}")


<a id="sec7"></a>
## 7. Summary and exercises

In this module we covered:
- the multiple comparisons problem as the reason NOT to run many t-tests at once,
- the logic of ANOVA: comparing between-group and within-group variance (the F
  statistic),
- one-way ANOVA via `scipy.stats.f_oneway` and `statsmodels`,
- ANOVA assumptions and Levene's test for multiple groups,
- the Tukey HSD post-hoc test for finding WHICH pairs of groups differ significantly,
- eta squared as a measure of effect size.

> 📝 **Exercise 1: ANOVA for spend by city**
>
> Run a one-way ANOVA checking whether `monthly_spend` differs significantly across the 5 cities. Interpret the result at `alpha=0.05`.

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
city_groups = [data["monthly_spend"].values for _, data in customers.groupby("city")]
f_cities, p_cities = stats.f_oneway(*city_groups)
print(f"F={f_cities:.3f}, p={p_cities:.4f}")
```

Spend in our data doesn't depend on city (only on a random right-skewed distribution), so expect no significance.
</details>

> 📝 **Exercise 2: Levene for cities**
>
> Check Levene's test for spend by city from exercise 1. Is the equal-variance assumption met?

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
print(stats.levene(*city_groups))
```
</details>

> 📝 **Exercise 3: Post-hoc for age by region**
>
> Run an ANOVA for `age` by `region`, and if it comes out significant, run Tukey HSD. If it does NOT come out significant, write in a comment why running a post-hoc test wouldn't make sense in that case.

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
age_groups = [data["age"].values for _, data in customers.groupby("region")]
f_age, p_age = stats.f_oneway(*age_groups)
print(f"F={f_age:.3f}, p={p_age:.4f}")
# age is drawn completely independently of region, so ANOVA will most likely NOT come
# out significant -- running Tukey HSD after a non-significant ANOVA is pointless,
# since ANOVA itself already says there's no evidence of ANY difference between groups
```
</details>

> 📝 **Exercise 4: Eta squared for spend by city**
>
> Compute eta squared for the model from exercise 1 (`monthly_spend ~ C(city)`, build the ANOVA table via `statsmodels` as in section 3). Compare the effect size with eta squared for region and satisfaction from section 6.

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
city_model = ols("monthly_spend ~ C(city)", data=customers).fit()
city_table = sm.stats.anova_lm(city_model, typ=2)
eta2_city = city_table["sum_sq"]["C(city)"] / city_table["sum_sq"].sum()
print(f"eta^2 (city): {eta2_city:.3f}")
print(f"eta^2 (region, section 6): {eta_squared:.3f}")
```
</details>

> 🔥 **Challenge: two-way ANOVA**
>
> Extend the model from section 3 with a SECOND factor: `satisfaction ~ C(region) + C(campaign_group)`. Build the ANOVA table (`sm.stats.anova_lm(model, typ=2)`) and check the rows for both factors separately. Does `campaign_group` contribute significantly to explaining satisfaction, on top of region?

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
two_way_model = ols("satisfaction ~ C(region) + C(campaign_group)", data=customers).fit()
two_way_table = sm.stats.anova_lm(two_way_model, typ=2)
two_way_table
```

The C(region) row should stay clearly significant (as in section 3). The C(campaign_group) row might surprise you: satisfaction in the data-generating code never depended on campaign group (only on spend and region) -- yet its p-value can come out below 0.05 anyway. That's not a data bug, it's a live example of the Type I error from Module 7: at alpha=0.05, even a TRULY zero effect will show up as falsely significant about once every 20 tries -- and this particular fixed random seed happened to land on exactly that.
</details>

## What's next?

ANOVA checks for DIFFERENCES between groups. In **Module 11** we'll shift perspective to
the **RELATIONSHIP** between two continuous variables: correlation and covariance -- the
foundation for the linear regression in Module 12, which will let us not just state a
relationship exists, but predict one variable from another.